## Setup note

This notebook reads unlabeled race photos from `../data/raw/`. Those images are not included in the repository (no redistribution license).

To run this notebook, populate `data/raw/` with any collection of race photos in jpg/jpeg/png format. The EDA methodology — perceptual hashing for deduplication, CLIP embeddings, UMAP diversity visualization, and image statistics — is dataset-agnostic.

Notebooks `03_training.ipynb` through `07_image_analysis.ipynb` use the labeled Roboflow dataset and do not depend on `data/raw/`.

# 01 — EDA: Image Grid
Load all raw image paths and display a grid to confirm everything reads cleanly.

In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

RAW_DIR = Path("../data/raw")
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

image_paths = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in EXTS])
print(f"{len(image_paths)} images found")
for p in image_paths:
    print(p.name)

In [ ]:
THUMB_SIZE = (200, 200)
COLS = 8

def load_thumb(path: Path) -> np.ndarray | None:
    try:
        img = Image.open(path).convert("RGB")
        img.thumbnail(THUMB_SIZE, Image.LANCZOS)
        return np.array(img)
    except Exception as e:
        print(f"FAILED: {path.name} — {e}")
        return None

thumbs = [(p, load_thumb(p)) for p in image_paths]
failed = [p.name for p, t in thumbs if t is None]
ok = [(p, t) for p, t in thumbs if t is not None]

print(f"{len(ok)} loaded, {len(failed)} failed")
if failed:
    print("Failed:", failed)

In [ ]:
rows = math.ceil(len(ok) / COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 2, rows * 2))
axes = np.array(axes).flatten()

for ax, (path, thumb) in zip(axes, ok):
    ax.imshow(thumb)
    ax.set_title(path.name[:18], fontsize=5)
    ax.axis("off")

for ax in axes[len(ok):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("image_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grid saved to notebooks/image_grid.png")

## Image Statistics — Brightness, Contrast, Resolution

Distributions should show meaningful spread. A tight cluster means the dataset lacks variety in that dimension.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

RAW_DIR = Path("../data/raw")
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}
image_paths = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in EXTS])

def image_stats(path: Path) -> dict:
    img = Image.open(path).convert("RGB")
    arr = np.array(img, dtype=np.float32)
    gray = arr.mean(axis=2)
    w, h = img.size
    return {
        "name": path.name,
        "width": w,
        "height": h,
        "megapixels": round(w * h / 1_000_000, 2),
        "brightness": gray.mean(),
        "contrast": gray.std(),
    }

stats = pd.DataFrame([image_stats(p) for p in image_paths])
stats.head()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(stats["brightness"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Brightness (mean pixel intensity)")
axes[0].set_xlabel("0 = black, 255 = white")

axes[1].hist(stats["contrast"], bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Contrast (pixel std dev)")
axes[1].set_xlabel("higher = more variation in luminance")

axes[2].hist(stats["megapixels"], bins=20, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Resolution (megapixels)")
axes[2].set_xlabel("width × height / 1M")

plt.tight_layout()
plt.savefig("stats_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(stats[["brightness", "contrast", "megapixels"]].describe().round(1))


## Embedding-Based Diversity — CLIP + UMAP

Run all images through CLIP to get 512-d embeddings, then reduce to 2D with UMAP.
Tight clusters indicate homogenous subsets; outliers are worth inspecting manually.

> **Runtime:** ~1–2 min on 127 images with MPS. CLIP model downloads on first run (~600 MB).

In [ ]:
from pathlib import Path
import numpy as np
import torch
import clip
from PIL import Image

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

RAW_DIR = Path("../data/raw")
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}
image_paths = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in EXTS])

model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()

def embed_images(paths: list[Path], batch_size: int = 16) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i : i + batch_size]
        tensors = []
        for p in batch:
            try:
                img = preprocess(Image.open(p).convert("RGB"))
                tensors.append(img)
            except Exception as e:
                print(f"SKIP {p.name}: {e}")
                tensors.append(torch.zeros_like(preprocess(Image.new("RGB", (224, 224)))))
        batch_tensor = torch.stack(tensors).to(DEVICE)
        with torch.no_grad():
            embeddings = model.encode_image(batch_tensor)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        all_embeddings.append(embeddings.cpu().float().numpy())
        print(f"  embedded {min(i + batch_size, len(paths))}/{len(paths)}")
    return np.vstack(all_embeddings)

embeddings = embed_images(image_paths)
print(f"Embeddings shape: {embeddings.shape}")


In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords = reducer.fit_transform(embeddings)

def infer_source(name: str) -> str:
    if name.startswith("Heartland"):   return "Heartland50"
    if name.startswith("LoudThunder"): return "LoudThunder"
    if name.startswith("Mile0"):       return "Mile0"
    if name.startswith("RockinK"):     return "RockinK"
    if name.startswith("IMG_"):        return "personal"
    if name.startswith("race_"):       return "runsignup"
    if name.startswith("Screenshot"):  return "screenshot"
    return "flickr/web"

sources = [infer_source(p.name) for p in image_paths]
unique_sources = sorted(set(sources))
palette = plt.cm.tab10.colors
color_map = {s: palette[i % len(palette)] for i, s in enumerate(unique_sources)}
colors = [color_map[s] for s in sources]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=30, alpha=0.8, linewidths=0)

for source in unique_sources:
    ax.scatter([], [], color=color_map[source], label=source, s=40)
ax.legend(title="Source", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.set_title("CLIP Embeddings — UMAP Projection")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig("clip_umap.png", dpi=150, bbox_inches="tight")
plt.show()
